# FttH aux Antilles : quelle vision sur la qualité de service, et où agir ?

**Cas d'étude Data Analyst** · Maritza Brival · Guadeloupe · novembre 2026

> ⚠️ **Données simulées, étiquetées comme telles.** L'Arcep ne publie encore aucun indicateur de qualité FttH pour les outre-mer. Ce cas construit l'observatoire qui manque : les dénominateurs (locaux, abonnés, raccordements) sont réels, les pannes et raccordements sont simulés à partir de repères nationaux sourcés et d'un dire d'expert terrain documenté.

Ce notebook **rejoue toute l'analyse** à partir de la version publiée du dépôt (tag `observatoire-ftth-qs-antilles_v1`) :

1. il régénère la base SQLite (schéma + données simulées, graine figée) ;
2. il recalcule les 8 indicateurs (+ 7b) et vérifie qu'ils sont **identiques** aux résultats publiés ;
3. il redessine les visuels du rapport.

- 📄 Rapport complet (démarche Ask → Act, contrôles, limites) : [PDF dans le dépôt](https://github.com/mdbrivalast/capstone-ftth-qs-antilles/blob/observatoire-ftth-qs-antilles_v1/docs/FTTH-Antilles-qualite-de-service.pdf)
- 💾 Code, requêtes et données : [dépôt GitHub](https://github.com/mdbrivalast/capstone-ftth-qs-antilles/tree/observatoire-ftth-qs-antilles_v1)
- ↩️ Cas d'étude précédent : [« Déployer ou faire adopter ? »](https://github.com/mdbrivalast/capstone-ftth-antilles)

*Prérequis Kaggle : Settings → Internet : ON (le notebook télécharge le dépôt).*

## La question

> Quels leviers peuvent être utilisés par la Région Guadeloupe et la CTM, sans attendre les données de l'Arcep et révélés par le modèle d'observatoire que je propose avec huit indicateurs, pour que les opérateurs concentrent leurs efforts sur la qualité de service ?

Périmètre : Martinique (972) et Guadeloupe, Saint-Barthélemy et Saint-Martin inclus (971 + 977 + 978), de janvier 2023 à décembre 2025 (36 mois, 3 saisons cycloniques).

## 1. Récupérer la version publiée du dépôt

In [ ]:
import subprocess, sys, os, sqlite3                 # subprocess : lancer git et les scripts ; sqlite3 : la base, livré avec Python
from pathlib import Path                             # Path : chemins de fichiers propres
import numpy as np                                   # numpy : comparaisons numériques avec tolérance
import pandas as pd                                  # pandas : lecture des CSV et des résultats SQL

TAG    = "observatoire-ftth-qs-antilles_v1"          # la baseline citée dans le rapport
URL    = "https://github.com/mdbrivalast/capstone-ftth-qs-antilles.git"
DEPOT  = Path("capstone-ftth-qs-antilles")           # dossier de travail (dans /kaggle/working sur Kaggle)

# On clone uniquement la version taguée (--depth 1 : sans l'historique, plus rapide)
if not DEPOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", TAG, URL, str(DEPOT)], check=True)

print("Dépôt prêt :", DEPOT.resolve())
print(sorted(p.name for p in DEPOT.iterdir()))

## 2. Régénérer la base

On part du schéma peuplé des référentiels (DDL v5.6 : causes, tronçons, parc annuel réel), puis le générateur simule les PM, les abonnés, les raccordements et les pannes. La graine est figée : le résultat est le même à chaque exécution.

In [ ]:
BASE = DEPOT / "cap3v5.db"                           # la base n'est pas versionnée : elle se régénère
if BASE.exists():
    BASE.unlink()                                    # on repart toujours d'une base neuve

# Étape 1 : schéma + tables de référence
con = sqlite3.connect(BASE)
con.executescript((DEPOT / "sql" / "DDL-capstone3-v5.6-peuple.sql").read_text(encoding="utf-8"))
con.commit()
con.close()

# Étape 2 : données simulées (environ 30 secondes)
res = subprocess.run([sys.executable, "scripts/generateur_faits_cap3.py", "cap3v5.db"],
                     cwd=DEPOT, capture_output=True, text=True)
print(res.stdout[-700:])                             # la fin du journal du générateur
res.check_returncode()                               # arrête le notebook si le générateur a échoué

In [ ]:
# Contrôle des volumes : ils doivent être identiques à ceux du rapport (section 3)
ATTENDU = {"dim_temps": 36, "dim_pm": 1580, "dim_abonne": 235000,
           "fait_raccordement": 181488, "fait_signalisation": 150527}

con = sqlite3.connect(BASE)
volumes = pd.DataFrame(
    [(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0], n) for t, n in ATTENDU.items()],
    columns=["table", "lignes", "attendu"])
volumes["contrôle"] = np.where(volumes["lignes"] == volumes["attendu"], "✅", "❌")
volumes

## 3. Recalculer les indicateurs et les comparer aux résultats publiés

Chaque requête de `sql/requetes/` est rejouée sur la base régénérée, puis comparée, valeur par valeur, au CSV publié à côté d'elle. Un ✅ veut dire que l'analyse est reproduite à l'identique.

In [ ]:
REQ = DEPOT / "sql" / "requetes"

def identiques(a, b):
    """Vrai si deux tableaux ont les mêmes colonnes et les mêmes valeurs
    (les nombres sont comparés avec une petite tolérance d'arrondi)."""
    if list(a.columns) != list(b.columns) or len(a) != len(b):
        return False
    a, b = a.reset_index(drop=True), b.reset_index(drop=True)
    for col in a.columns:
        x, y = pd.to_numeric(a[col], errors="coerce"), pd.to_numeric(b[col], errors="coerce")
        if x.notna().all() and y.notna().all():                     # colonne numérique
            if not np.allclose(x, y, atol=0.051):                   # tolérance : un arrondi à 0,1 près
                return False
        elif not (a[col].astype(str) == b[col].astype(str)).all():  # colonne texte
            return False
    return True

resultats, bilan = {}, []
for fichier in sorted(REQ.glob("req_indicateur*.sql")):
    nom = fichier.stem.replace("req_", "")                          # ex. indicateur04_taux_pannes
    calcule = pd.read_sql(fichier.read_text(encoding="utf-8"), con) # la requête rejouée
    publie  = pd.read_csv(REQ / f"res_{nom}.csv", sep=";",
                          keep_default_na=False)                    # le résultat publié ; keep_default_na=False : le segment « NA » (Transverse) reste du texte, pandas le lirait sinon comme une valeur manquante
    resultats[nom] = calcule
    # tri commun avant comparaison (l'ordre des lignes peut varier d'un export à l'autre)
    cle = calcule.columns[0]
    ok = identiques(calcule.sort_values(cle), publie.sort_values(cle))
    bilan.append((nom, len(calcule), "✅" if ok else "❌"))

pd.DataFrame(bilan, columns=["indicateur", "lignes", "identique au publié"])

In [ ]:
# Quelques résultats clés, tels que cités dans le rapport
for nom in ["indicateur01_raccordement_issue", "indicateur06_reiteration",
            "indicateur07_malfacon-cause", "indicateur08_cyclone_tranche-aerienpm"]:
    print(f"\n— {nom}")
    display(resultats[nom])

## 4. Redessiner les visuels

Les scripts de `scripts/visuels/` lisent les résultats publiés (et la base pour le visuel 4) et écrivent leurs images dans `images/`. La planche réunit la tuile KPI, les 7 visuels et une note de méthode.

In [ ]:
from IPython.display import SVG, display, Markdown

os.environ["MPLBACKEND"] = "Agg"                     # dessin sans fenêtre (les images sont enregistrées)
SCRIPTS = ["v0_tuile_reiteration", "v1_parite_entree", "v23_taux_vs_volume", "v4_mttr_segment",
           "v5_pannes_malfacon", "v6_cyclone_aerien", "v7_dilution_tammy", "planche_share"]
for s in SCRIPTS:
    subprocess.run([sys.executable, f"scripts/visuels/{s}.py"], cwd=DEPOT, check=True)
print("Visuels redessinés dans", (DEPOT / "images").resolve())

In [ ]:
# La planche récapitulative
display(SVG(filename=str(DEPOT / "images" / "planche_share.svg")))

In [ ]:
# Les visuels un par un, avec leur message
MESSAGES = {
    "v0_tuile_reiteration": "32,7 % des abonnés touchés par une panne en ont connu au moins deux.",
    "v1_parite_entree":     "Parité des deux territoires à l'entrée, en retrait du national.",
    "v23_taux_vs_volume":   "Taux ou volume : deux lectures du même réseau.",
    "v4_mttr_segment":      "Plus la panne est proche de l'abonné, plus la réparation est longue.",
    "v5_pannes_malfacon":   "Près d'une panne sur trois vient d'une malfaçon.",
    "v6_cyclone_aerien":    "Plus le PM est aérien, plus l'impact cyclonique est important.",
    "v7_dilution_tammy":    "Pendant Tammy, les malfaçons stagnent, les autres pannes explosent.",
}
for s, message in MESSAGES.items():
    display(Markdown(f"**{message}**"))
    display(SVG(filename=str(DEPOT / "images" / f"{s}.svg")))

In [ ]:
con.close()                                          # on referme proprement la base

## 5. Ce que l'observatoire montre, et où agir

- **Une qualité en retrait dès l'entrée** : 11 % d'échecs au raccordement contre 6,27 % au national, à l'identique dans les deux territoires.
- **Deux causes distinctes** : la malfaçon, chronique, à l'origine de près d'une panne sur trois (29,4 %) ; un réseau très aérien, que les cyclones frappent presque deux fois plus fort (×1,91 sur les PM aériens pendant Tammy, contre ×1,40 sur les PM enterrés).
- **Deux leviers pour la Région Guadeloupe et la CTM** : la qualité du travail, en s'adressant au bon acteur selon la nature de la cause (OI, STOC, OC) ; la résilience de l'aérien, suivie par un seuil du type « 75 % des pannes réparées en moins de X jours » plutôt que par la moyenne.

**Limite principale** : les données sont simulées. Le simulateur restitue ce qu'on lui a injecté ; il ne confirme pas les hypothèses, il montre qu'elles sont cohérentes entre elles et qu'on pourra les vérifier dès que l'Arcep publiera les données des outre-mer. Le détail des contrôles et des limites est dans le [rapport complet](https://github.com/mdbrivalast/capstone-ftth-qs-antilles/blob/observatoire-ftth-qs-antilles_v1/docs/FTTH-Antilles-qualite-de-service.pdf).